In [19]:
import fiftyone
import fiftyone.utils.random
import pathlib
import yaml
from ultralytics import YOLO


# set up the paths, names and settings

In [20]:
# select to models parent folder
models_parent_dir = pathlib.Path.cwd().parent / "models"
datasets_parent_dir = pathlib.Path.cwd().parent / "datasets"

# choose a name for the new merged dataset
merged_dataset_name = "der_merger"

# Load settings
with open('settings.yaml') as f:
    settings = yaml.safe_load(f)

splits = ["train", "test", "val"]

# All splits must use the same classes list
classes_to_export = ["Chair", "Window", "Light", "Door"]

# if the predictions labels are not to your liking
label_remapping = {"chair": "Chair", "window": "Window", "light": "Light"}

# choose models to inference with, they must match the settings.yaml
chosen_models = ["doors", "windows", "lights"]

# setup the datasets in fiftyone

In [21]:
datasets = fiftyone.list_datasets()
print(datasets)

['coco-2017-train-validation-test', 'coco-2017-train-validation-test-200', 'der_merger', 'der_merger-57422', 'ebi-fused-4', 'open-images-v7-train-validation-test', 'open-images-v7-train-validation-test-200', 'tmp_dataset']


In [22]:

try:
    tmp_dataset = fiftyone.load_dataset("tmp_dataset")
except:
    tmp_dataset = fiftyone.Dataset("tmp_dataset")
tmp_dataset.clear()

# choose a dataset to add the predictions to
source_dataset = fiftyone.load_dataset("coco-2017-train-validation-test")
tmp_dataset.add_samples(source_dataset)

# datasets_to_add = [fiftyone.load_dataset("ebi-fused-4")]
datasets_to_add = [fiftyone.load_dataset("ebi-fused-4")]

 100% |█████████████| 54024/54024 [2.7m elapsed, 0s remaining, 1.7K samples/s]      


# model inferencing on a particular subset

In [23]:

conditions = []

# Apply all models to same field, then filter once
for model_name in chosen_models:
    model = YOLO(models_parent_dir / model_name / "best.pt")
    tmp_dataset.apply_model(model, label_field=model_name)

    for class_name, threshold in settings["models"][model_name].items():
        conditions.append(
            (fiftyone.ViewField("label") == class_name) & 
            (fiftyone.ViewField("confidence") >= threshold)
        )
    if conditions:
        keep_condition = conditions[0]
        for condition in conditions[1:]:
            keep_condition |= condition
        filtered_view = tmp_dataset.filter_labels(model_name, keep_condition)
        filtered_view.merge_labels(in_field=model_name, out_field="ground_truth")


# Delete predictions field from the original dataset
# tmp_dataset.delete_sample_fields("predictions")
tmp_dataset.save()

 100% |█████████████| 54024/54024 [10.0m elapsed, 0s remaining, 78.6 samples/s]      
 100% |█████████████| 54024/54024 [10.1m elapsed, 0s remaining, 86.1 samples/s]      
 100% |█████████████| 54024/54024 [10.2m elapsed, 0s remaining, 86.4 samples/s]      


In [24]:
# session = fiftyone.launch_app(tmp_dataset)

# merge datasets

In [25]:
datasets_to_add.append(tmp_dataset)

size = 0

for dataset in datasets_to_add:
    size += len(dataset)


try:
    merged_dataset  = fiftyone.load_dataset(f"{merged_dataset_name}-{size}")
except:
    merged_dataset  = fiftyone.Dataset(f"{merged_dataset_name}-{size}")
merged_dataset .clear()
print(f"dataset name: {merged_dataset_name}-{size}")

for dataset in datasets_to_add:
    print(f"adding {len(dataset)} samples from {dataset.name}")
    merged_dataset.merge_samples(dataset)

merged_dataset.save()
print(merged_dataset)

dataset name: der_merger-57422
adding 3398 samples from ebi-fused-4
adding 54024 samples from tmp_dataset
Name:        der_merger-57422
Media type:  image
Num samples: 57422
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    doors:            fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    windows:          fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    lights:           fiftyone.core.fields.EmbeddedDocumentField(f

# some tag renaming and splits redistribution

In [26]:
# Find all samples tagged with "validation" and retag them as "val"
validation_samples = merged_dataset.match_tags("validation")
validation_samples.tag_samples("val")
validation_samples.untag_samples("validation")


print(f"Splits before redistribution: {merged_dataset.count_sample_tags()}")
merged_dataset.untag_samples(splits)
fiftyone.utils.random.random_split(merged_dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(f"Splits after redistribution: {merged_dataset.count_sample_tags()}")

Splits before redistribution: {'train': 15494, 'val': 915, 'test': 41013}
Splits after redistribution: {'test': 11485, 'val': 5742, 'train': 40195}


# some class renaming

In [27]:
# session = fiftyone.Session(merged_dataset)

In [28]:
label_fields = merged_dataset._get_label_fields()
for field in label_fields:
    view = merged_dataset.map_labels(field, label_remapping)
    merged_dataset = view
    

In [29]:

# merged_dataset = merged_dataset.take(400)

export_dir = datasets_parent_dir / "multiclass" / f"{merged_dataset_name}-{len(merged_dataset)}"

# Export the splits
for split in splits:
    split_view = merged_dataset.match_tags(split)
    split_view.export(
        export_dir=str(export_dir),
        dataset_type=fiftyone.types.YOLOv5Dataset,
        label_field="ground_truth",
        # overwrite=True,
        split=split,
        classes=classes_to_export,
        progress=True,
    )


   6% |-------------|  2445/40195 [3.9s elapsed, 59.7s remaining, 741.6 samples/s]    

/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'fork' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'knife' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'pizza' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'dining table' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'person' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/on

   6% ||------------|  2501/40195 [4.1s elapsed, 1.0m remaining, 617.9 samples/s]     

/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'hot dog' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'baseball bat' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'baseball glove' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'suitcase' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'sheep' not in provided classes
  warnings.warn(msg)
/home/rolf/ana

   6% |\------------|  2591/40195 [4.4s elapsed, 1.1m remaining, 426.5 samples/s]     

...

  10% |█\-----------|  3872/40195 [8.9s elapsed, 1.5m remaining, 280.8 samples/s]     

/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'hair drier' not in provided classes
  warnings.warn(msg)


  11% |█|-----------|  4469/40195 [11.1s elapsed, 1.5m remaining, 283.9 samples/s]    

/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'cow' not in provided classes
  warnings.warn(msg)


 100% |█████████████| 40195/40195 [1.1m elapsed, 0s remaining, 1.1K samples/s]        
Directory '/home/rolf/GIT/onemanstreasure/datasets/multiclass/der_merger-57422' already exists; export will be merged with existing files
 100% |█████████████| 11485/11485 [19.2s elapsed, 0s remaining, 1.0K samples/s]       
Directory '/home/rolf/GIT/onemanstreasure/datasets/multiclass/der_merger-57422' already exists; export will be merged with existing files
 100% |███████████████| 5742/5742 [9.6s elapsed, 0s remaining, 1.0K samples/s]        
